# Auto-Labeling Pipeline — Stage 1: Image-Only, Single Frame, CAM_FRONT

**Modules:**
1. SAM3 segmentation (inline text-prompted, all object classes)
2. SAM3D Body inference (pedestrians)
3. SAM3D Objects inference (all other classes, seeded with SAM3 masks + MoGe pointmap)
4. Orientation estimation (`facing_direction` for pedestrians; ambiguous long axis for objects)
5. OBB fitting (oriented bounding box in ego frame)
6. 3D visualization (k3d)

**Assumptions for stage 1:**
- No LiDAR integration (image-only; MoGe depth is relative/non-metric)
- No ground removal
- Single frame, no temporal context
- Object yaw has 180° ambiguity (pedestrians are fully resolved via skeleton)

## 0. Imports

In [ ]:
import sys
import os
import shutil
import tempfile
from pathlib import Path

import numpy as np
import cv2
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from pyquaternion import Quaternion
from nuscenes.nuscenes import NuScenes

import plotly.graph_objects as go

## 1. Config

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Dataset selector  →  change this one variable to switch datasets
# ══════════════════════════════════════════════════════════════════════════════
DATASET = 'ecp'           # 'ecp'  or  'nuscenes_mini'

# ── Per-dataset defaults ───────────────────────────────────────────────────────
if DATASET == 'nuscenes_mini':
    DATA_ROOT    = '/media/lleba/nuScenes_mini'
    NUSCENES_VER = 'v1.0-mini'
    SCENE_IDX    = 0     # mini has 10 scenes (0–9)
    FRAME_IDX    = 0
elif DATASET == 'ecp':
    DATA_ROOT    = '/media/lleba/ECP_Nuscenes_01/output2/ecp2nuscenes'
    NUSCENES_VER = 'v1.0-trainval'
    SCENE_IDX    = 10    # ECP scene 10, frame 570 used for development
    FRAME_IDX    = 570
else:
    raise ValueError(f"Unknown DATASET: {DATASET!r}  (must be 'ecp' or 'nuscenes_mini')")

CAMERA = 'CAM_FRONT'

# ── Model checkpoints ─────────────────────────────────────────────────────────
DEV_ROOT = Path('/home/lleba/Thesis/Development')

# SAM3 checkpoint resolved via HuggingFace Hub (downloads on first run, cached after)
from huggingface_hub import hf_hub_download
SAM3_CKPT = hf_hub_download(repo_id='facebook/sam3', filename='sam3.pt')

SAM3D_BODY_REPO = DEV_ROOT / 'SAM3D' / 'sam-3d-body'
SAM3D_BODY_CKPT = SAM3D_BODY_REPO / 'checkpoints' / 'sam-3d-body-dinov3' / 'model.ckpt'
SAM3D_BODY_MHR  = SAM3D_BODY_REPO / 'checkpoints' / 'sam-3d-body-dinov3' / 'assets' / 'mhr_model.pt'

SAM3D_OBJ_REPO  = DEV_ROOT / 'SAM3D' / 'sam-3d-objects'
SAM3D_OBJ_CFG   = SAM3D_OBJ_REPO / 'checkpoints' / 'hf' / 'pipeline.yaml'
# inference.py is in the notebook/ subfolder — it sets LIDRA_SKIP_INIT before any sam3d_objects import

# ── SAM3 text prompts per class ───────────────────────────────────────────────
# Maps text prompt → which sub-pipeline handles it
# 'body'    → SAM3D Body (pedestrians)
# 'objects' → SAM3D Objects (vehicles, cyclists, ...)
SAM3_PROMPTS = {
    'pedestrian':           'body',
    'car':                  'objects',
    'bicycle':              'objects',
    'motorcycle':           'objects',
    'bus':                  'objects',
    'truck':                'objects',
    'construction vehicle': 'objects',
    'trailer':              'objects',
}

# ── Detection thresholds ──────────────────────────────────────────────────────
SAM3_SCORE_THRESH     = 0.5    # min SAM3 mask probability to keep
MIN_MASK_PX           = 100    # min mask area in pixels
BODY_BBOX_THRESH      = 0.8    # ViTDet confidence threshold for SAM3D Body
SAM3_IOU_MERGE_THRESH = 0.4    # IoU above which a ViTDet box is a duplicate of a SAM3 box

# ── Pointmap mode ─────────────────────────────────────────────────────────────
# Controls what depth information is fed into SAM3D Objects (Section 6).
#
#   'baseline'       — dense MoGe relative-depth pointmap (no LiDAR). Original pipeline.
#   'o1_lidar'       — sparse metric LiDAR pointmap. LiDAR replaces MoGe entirely.
#                      Most pixels are empty (NaN); model fills gaps from image priors.
#   'o2_moge_affine' — dense MoGe pointmap calibrated to metric scale via a global
#                      affine fit  Z_metric = a*Z_moge + b  (least-squares on all
#                      in-image LiDAR returns). Every pixel gets a metric depth estimate.
#   'o3_local_affine'— dense MoGe pointmap with a per-object affine fit. For each
#                      detected object, HDBSCAN filters its in-mask LiDAR surface
#                      points in 3D ego space, then fits a separate (a,b) for that
#                      object. Full-frame pointmap is rebuilt per object before each
#                      SAM3D Objects call. Falls back to O2 if too few clean points.
#
POINTMAP_MODE = 'o3_local_affine'  # 'baseline' | 'o1_lidar' | 'o2_moge_affine' | 'o3_local_affine'

print(f'Dataset: {DATASET}  |  scene={SCENE_IDX}  frame={FRAME_IDX}  cam={CAMERA}')
print(f'POINTMAP_MODE={POINTMAP_MODE!r}')
print(f'SAM3 checkpoint: {SAM3_CKPT}')


## 2. Path Setup
Models are loaded sequentially — each section loads its model, runs inference, then unloads before the next model is loaded. This keeps peak GPU memory to one model at a time.

In [ ]:
import gc

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_float32_matmul_precision('high')

# Add all repo paths once here so each section can import without repeating sys.path inserts
sys.path.insert(0, str(DEV_ROOT / 'SAM3'))
sys.path.insert(0, str(SAM3D_BODY_REPO))
sys.path.insert(0, str(SAM3D_OBJ_REPO / 'notebook'))
os.environ['LIDRA_SKIP_INIT'] = 'true'

print('Paths configured. Models will be loaded sequentially in each section.')

## 3. Data Loading
Load the selected frame: image, camera intrinsics, and camera-to-ego extrinsics.

In [ ]:
def load_frame(nusc, scene_idx, frame_idx, camera='CAM_FRONT'):
    """
    Load image, intrinsics, and camera-to-ego extrinsics for one frame.

    Returns
    -------
    img_bgr       : (H, W, 3) uint8 BGR image
    img_rgb       : (H, W, 3) uint8 RGB image
    K             : (3, 3) float64 camera intrinsics
    R_c2e         : (3, 3) float64 rotation camera → ego
    t_c2e         : (3,)   float64 translation camera → ego
    img_path      : str    absolute path to image file
    cam_sd_token  : str    nuScenes sample_data token for this camera frame
    """
    scene   = nusc.scene[scene_idx]
    token   = nusc.get('sample', scene['first_sample_token'])['data'][camera]
    for _ in range(frame_idx):
        rec   = nusc.get('sample_data', token)
        token = rec['next']
        assert token, f'frame_idx {frame_idx} exceeds scene length'

    rec      = nusc.get('sample_data', token)
    img_path = nusc.get_sample_data_path(token)
    cal      = nusc.get('calibrated_sensor', rec['calibrated_sensor_token'])

    K     = np.array(cal['camera_intrinsic'], dtype=np.float64)
    R_c2e = Quaternion(cal['rotation']).rotation_matrix.astype(np.float64)
    t_c2e = np.array(cal['translation'], dtype=np.float64)

    img_bgr = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    return img_rgb, img_bgr, K, R_c2e, t_c2e, img_path, token


nusc    = NuScenes(version=NUSCENES_VER, dataroot=DATA_ROOT, verbose=False)
img_rgb, img_bgr, K, R_c2e, t_c2e, img_path, cam_sd_token = load_frame(
    nusc, SCENE_IDX, FRAME_IDX, CAMERA
)
H, W = img_rgb.shape[:2]
fx, fy, cx, cy = K[0,0], K[1,1], K[0,2], K[1,2]

print(f'Image: {img_path}')
print(f'  Size : {W}×{H}')
print(f'  fx={fx:.1f}  fy={fy:.1f}  cx={cx:.1f}  cy={cy:.1f}')

plt.figure(figsize=(14, 5))
plt.imshow(img_rgb)
plt.axis('off')
plt.title(f'Scene {SCENE_IDX}, frame {FRAME_IDX}, {CAMERA}')
plt.tight_layout()
plt.show()

## 3.5 LiDAR Loading and Camera Projection
Load the timestamp-synced LIDAR_TOP sweep and project visible points into the camera frame.
Produces `pts_ego_vis`, `pts_cam_vis`, `u_vis`, `v_vis`, `Z_vis` used downstream by O1.

In [ ]:
# ── Find timestamp-synced LIDAR_TOP ──────────────────────────────────────────
cam_sd = nusc.get('sample_data', cam_sd_token)

if cam_sd['is_key_frame']:
    _sample     = nusc.get('sample', cam_sd['sample_token'])
    lidar_token = _sample['data']['LIDAR_TOP']
else:
    _cam_ts     = cam_sd['timestamp']
    lidar_token = min(
        (sd for sd in nusc.sample_data if sd['channel'] == 'LIDAR_TOP'),
        key=lambda sd: abs(sd['timestamp'] - _cam_ts),
    )['token']

lidar_sd   = nusc.get('sample_data', lidar_token)
lidar_path = nusc.get_sample_data_path(lidar_token)
lidar_cal  = nusc.get('calibrated_sensor', lidar_sd['calibrated_sensor_token'])
R_l2e      = Quaternion(lidar_cal['rotation']).rotation_matrix.astype(np.float64)
t_l2e      = np.array(lidar_cal['translation'], dtype=np.float64)
pts_raw    = np.fromfile(lidar_path, dtype=np.float32).reshape(-1, 5)[:, :3]
pts_ego    = (R_l2e @ pts_raw.astype(np.float64).T).T + t_l2e
print(f'LiDAR: {len(pts_ego):,} points  ({lidar_path})')

# ── Project LiDAR points into camera frame ────────────────────────────────────
# ego → camera (R3/OpenCV): R_c2e^T @ (p_ego - t_c2e)
pts_cam = (R_c2e.T @ (pts_ego - t_c2e).T).T   # (N, 3)  R3 camera frame

Z_cam   = pts_cam[:, 2]
front   = Z_cam > 0
u_proj  = pts_cam[front, 0] / Z_cam[front] * fx + cx
v_proj  = pts_cam[front, 1] / Z_cam[front] * fy + cy
in_img  = (u_proj >= 0) & (u_proj < W) & (v_proj >= 0) & (v_proj < H)

pts_cam_vis = pts_cam[front][in_img].astype(np.float32)   # (M, 3)  R3 camera coords
pts_ego_vis = pts_ego[front][in_img].astype(np.float32)   # (M, 3)  ego coords
u_vis       = u_proj[in_img].astype(np.float32)           # (M,)    pixel column
v_vis       = v_proj[in_img].astype(np.float32)           # (M,)    pixel row
Z_vis       = pts_cam_vis[:, 2]                           # (M,)    depth (metres)

print(f'Visible LiDAR pts in {CAMERA}: {len(pts_cam_vis):,}  '
      f'(depth range: {Z_vis.min():.1f}–{Z_vis.max():.1f} m)')

## 4. SAM3 Segmentation
Run SAM3 inline with a text prompt for each class. A separate session is opened per class (same pattern as `get_sam3_masks_for_nuscenes_trainval.py`).

In [ ]:
from sam3.model_builder import build_sam3_video_predictor

print('Loading SAM3...')
gpus = range(torch.cuda.device_count())
sam3_predictor = build_sam3_video_predictor(gpus_to_use=gpus, checkpoint_path=str(SAM3_CKPT))
print('  SAM3 ready.')


def run_sam3_single_frame(predictor, img_rgb, text_prompt, score_thresh=0.5, min_mask_px=100):
    """
    Run SAM3 with a text prompt on a single frame.

    The image is written to a temp directory as a single-frame 'video',
    then SAM3 propagates over it and the session is closed.

    Parameters
    ----------
    predictor    : SAM3 video predictor instance
    img_rgb      : (H, W, 3) uint8 RGB image
    text_prompt  : str  e.g. 'car', 'pedestrian', 'bicycle and cyclist'
    score_thresh : float  min object probability to keep
    min_mask_px  : int    min mask area to keep

    Returns
    -------
    detections : list of dicts, each with:
        'binary_mask'  : (H, W) bool
        'score'        : float
        'box_xywh'     : (4,) float  normalised [x, y, w, h]
        'prompt'       : str
    """
    tmp_dir = tempfile.mkdtemp(prefix='sam3_frame_')
    try:
        Image.fromarray(img_rgb).save(os.path.join(tmp_dir, '00000.jpg'))
        resp       = predictor.handle_request(dict(type='start_session', resource_path=tmp_dir))
        session_id = resp['session_id']
        predictor.handle_request(dict(
            type='add_prompt', session_id=session_id, frame_index=0, text=text_prompt
        ))
        frame_outputs = {}
        with torch.autocast('cuda', dtype=torch.bfloat16):
            for resp in predictor.handle_stream_request(dict(
                type='propagate_in_video', session_id=session_id
            )):
                frame_outputs[resp['frame_index']] = resp['outputs']
        predictor.handle_request(dict(type='close_session', session_id=session_id))
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

    out      = frame_outputs.get(0, {})
    obj_ids  = out.get('out_obj_ids', [])
    probs    = np.asarray(out.get('out_probs', []), dtype=np.float32)
    masks    = out.get('out_binary_masks', [])
    boxes    = np.asarray(out.get('out_boxes_xywh', []), dtype=np.float32)

    detections = []
    for i, (oid, prob) in enumerate(zip(obj_ids, probs)):
        if prob < score_thresh:
            continue
        bm = np.asarray(masks[i], dtype=bool)
        if bm.sum() < min_mask_px:
            continue
        detections.append({
            'binary_mask': bm,
            'score':       float(prob),
            'box_xywh':    boxes[i] if len(boxes) > i else None,
            'prompt':      text_prompt,
        })
    return detections


# Run SAM3 for all classes
sam3_results = {}
for prompt, pipeline in SAM3_PROMPTS.items():
    print(f'Running SAM3: "{prompt}" ...', end=' ')
    dets = run_sam3_single_frame(
        sam3_predictor, img_rgb, prompt,
        score_thresh=SAM3_SCORE_THRESH, min_mask_px=MIN_MASK_PX
    )
    sam3_results[prompt] = dets
    print(f'{len(dets)} detection(s)')

total = sum(len(v) for v in sam3_results.values())
print(f'\nTotal detections: {total}')

In [ ]:
# ── Cross-class deduplication ─────────────────────────────────────────────────
# When SAM3 runs each class separately, the same physical object can receive
# masks from two prompts.  We resolve this via a strictly unique 8-tier
# priority hierarchy — every pair has a definite winner.
#
#   0  pedestrian            generic person
#   1  car                   generic vehicle
#   2  bicycle and cyclist   unpowered rider
#   3  truck                 specific vehicle; beaten by bus/constr./trailer
#   4  motorcycle            motorised rider > bicycle
#   5  bus                   large vehicle > truck
#   6  construction vehicle  heavy equipment > bus
#   7  trailer               most specific (semi-trailer > truck)
#
# Masks with IoU > CROSS_CLASS_IOU_THRESH from different classes are compared;
# the lower-priority detection is removed.

CROSS_CLASS_PRIORITY = {
    'pedestrian':           0,
    'car':                  1,
    'bicycle and cyclist':  2,
    'truck':                3,
    'motorcycle':           4,
    'trailer':              5,
    'bus':                  6,
    'construction vehicle': 7,
}
CROSS_CLASS_IOU_THRESH = 0.5


def dedup_cross_class(results, iou_thresh=CROSS_CLASS_IOU_THRESH, priority=CROSS_CLASS_PRIORITY):
    flat = [(p, d) for p, dets in results.items() for d in dets]
    keep = [True] * len(flat)
    for i in range(len(flat)):
        for j in range(i + 1, len(flat)):
            if not keep[i] or not keep[j]:
                continue
            p_i, d_i = flat[i]
            p_j, d_j = flat[j]
            if p_i == p_j:
                continue
            a, b  = d_i['binary_mask'], d_j['binary_mask']
            inter = np.logical_and(a, b).sum()
            if inter == 0:
                continue
            iou = inter / np.logical_or(a, b).sum()
            if iou < iou_thresh:
                continue
            if priority.get(p_i, 0) >= priority.get(p_j, 0):
                keep[j] = False
            else:
                keep[i] = False

    out = {p: [] for p in results}
    for (p, d), k in zip(flat, keep):
        if k:
            out[p].append(d)

    removed = sum(1 for k in keep if not k)
    print(f'Cross-class dedup: removed {removed} duplicate mask(s)  (IoU thresh={iou_thresh})')
    for p, dets in out.items():
        if dets:
            print(f'  {p}: {len(dets)}')
    return out


sam3_results = dedup_cross_class(sam3_results)
total_after  = sum(len(v) for v in sam3_results.values())
print(f'Detections after dedup: {total_after}')

In [ ]:
# ── Class colour map (one fixed colour per prompt) ────────────────────────────
CLASS_COLORS = {
    'pedestrian':           np.array([255,  60,  60], dtype=np.float32),
    'car':                  np.array([ 60, 180,  60], dtype=np.float32),
    'bicycle and cyclist':  np.array([ 60, 120, 255], dtype=np.float32),
    'motorcycle':           np.array([255, 160,   0], dtype=np.float32),
    'bus':                  np.array([180,  60, 255], dtype=np.float32),
    'truck':                np.array([  0, 210, 210], dtype=np.float32),
    'construction vehicle': np.array([255,  80, 180], dtype=np.float32),
    'trailer':              np.array([140, 200,  60], dtype=np.float32),
}
OUTLINE_KERNEL = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))  # outline thickness

overlay = img_rgb.copy().astype(np.float32)
outline_layer = np.zeros((H, W), dtype=bool)   # accumulate all outlines

# First pass: fill colours
for prompt, dets in sam3_results.items():
    col = CLASS_COLORS.get(prompt, np.array([200, 200, 200], dtype=np.float32))
    for d in dets:
        mask = d['binary_mask']
        overlay[mask] = overlay[mask] * 0.45 + col * 0.55

# Second pass: compute and draw outlines (dilated mask XOR original = boundary ring)
for prompt, dets in sam3_results.items():
    for d in dets:
        mask_u8  = d['binary_mask'].astype(np.uint8) * 255
        dilated  = cv2.dilate(mask_u8, OUTLINE_KERNEL)
        outline  = (dilated > 0) & ~d['binary_mask']
        outline_layer |= outline

# Draw outlines in white on top of colour overlay
overlay[outline_layer] = np.array([255, 255, 255], dtype=np.float32)

# Build legend (one entry per class that has detections)
legend_patches = []
for prompt, dets in sam3_results.items():
    if not dets:
        continue
    col = CLASS_COLORS.get(prompt, np.array([200, 200, 200], dtype=np.float32))
    legend_patches.append(
        patches.Patch(color=col / 255, label=f'{prompt} ({len(dets)})')
    )

fig, ax = plt.subplots(1, 1, figsize=(14, 5))
ax.imshow(overlay.astype(np.uint8))
ax.axis('off')
ax.set_title('SAM3 detections (all classes)')
if legend_patches:
    ax.legend(handles=legend_patches, loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Unload SAM3 before loading the next model
del sam3_predictor
gc.collect()
torch.cuda.empty_cache()
print(f'SAM3 unloaded. GPU free: {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB')

## 5. SAM3D Body — Pedestrian Meshes
Uses the internal ViTDet detector supplemented with SAM3 pedestrian detections.

In [ ]:
from sam_3d_body import load_sam_3d_body, SAM3DBodyEstimator
from tools.build_detector import HumanDetector

# SAM3 loading leaves torch.autocast globally enabled (BF16).
# Explicitly disable it before loading SAM3D Body so model init and inference
# are not affected. process_one_image is also wrapped below for safety.
print(f'Autocast state inherited from SAM3: {torch.is_autocast_enabled()}')
torch.autocast('cuda', enabled=False).__enter__()
print(f'Autocast disabled: {torch.is_autocast_enabled()}')

print('Loading SAM3D Body...')
_body_model, _body_cfg = load_sam_3d_body(
    str(SAM3D_BODY_CKPT), device=device, mhr_path=str(SAM3D_BODY_MHR)
)
body_estimator = SAM3DBodyEstimator(
    sam_3d_body_model=_body_model,
    model_cfg=_body_cfg,
    human_detector=HumanDetector(name='vitdet', device=device),
    human_segmentor=None,
    fov_estimator=None,
)
print('  SAM3D Body ready.')


def merge_with_vitdet(sam3_boxes, vitdet_boxes, iou_thresh=0.4):
    """
    Return SAM3 boxes plus any ViTDet box not already covered by a SAM3 box.

    SAM3 boxes take priority; ViTDet only fills gaps (IoU < iou_thresh with all SAM3 boxes).
    """
    if len(vitdet_boxes) == 0:
        return sam3_boxes
    if len(sam3_boxes) == 0:
        return vitdet_boxes
    ix1   = np.maximum(vitdet_boxes[:, 0:1], sam3_boxes[:, 0])
    iy1   = np.maximum(vitdet_boxes[:, 1:2], sam3_boxes[:, 1])
    ix2   = np.minimum(vitdet_boxes[:, 2:3], sam3_boxes[:, 2])
    iy2   = np.minimum(vitdet_boxes[:, 3:4], sam3_boxes[:, 3])
    inter = np.maximum(0.0, ix2 - ix1) * np.maximum(0.0, iy2 - iy1)
    area_v = ((vitdet_boxes[:, 2] - vitdet_boxes[:, 0]) *
              (vitdet_boxes[:, 3] - vitdet_boxes[:, 1]))[:, None]
    area_s = ((sam3_boxes[:, 2] - sam3_boxes[:, 0]) *
              (sam3_boxes[:, 3] - sam3_boxes[:, 1]))[None, :]
    iou   = inter / (area_v + area_s - inter + 1e-8)   # (K_vitdet, M_sam3)
    extra = vitdet_boxes[iou.max(axis=1) < iou_thresh]
    if len(extra) == 0:
        return sam3_boxes
    return np.vstack([sam3_boxes, extra])


def run_sam3d_body(estimator, img_path, img_bgr, K, sam3_ped_dets,
                   bbox_thresh=0.8, iou_merge_thresh=0.4):
    """
    Run SAM3D Body on one image.

    SAM3 pedestrian masks → pixel boxes (primary source).
    ViTDet is run separately and merged in: any ViTDet box with IoU < iou_merge_thresh
    against all SAM3 boxes is kept as a supplementary detection.
    """
    K_t = torch.tensor(K, dtype=torch.float32).unsqueeze(0)

    # SAM3 masks → pixel-space xyxy boxes
    sam3_boxes = []
    for d in sam3_ped_dets:
        ys, xs = np.where(d['binary_mask'])
        if len(xs) == 0:
            continue
        sam3_boxes.append([xs.min(), ys.min(), xs.max(), ys.max()])
    sam3_boxes = (np.array(sam3_boxes, dtype=np.float32)
                  if sam3_boxes else np.empty((0, 4), dtype=np.float32))

    # Run ViTDet separately and merge
    vitdet_boxes = estimator.detector.run_human_detection(
        img_bgr, bbox_thr=bbox_thresh, nms_thr=0.3, default_to_full_image=False
    )
    vitdet_boxes = np.asarray(vitdet_boxes, dtype=np.float32).reshape(-1, 4)
    merged_boxes = merge_with_vitdet(sam3_boxes, vitdet_boxes, iou_thresh=iou_merge_thresh)
    print(f'    SAM3: {len(sam3_boxes)}  ViTDet: {len(vitdet_boxes)}  merged: {len(merged_boxes)}')

    if len(merged_boxes) == 0:
        return []

    with torch.autocast('cuda', enabled=False):
        outputs = estimator.process_one_image(
            img_path, bboxes=merged_boxes, bbox_thr=bbox_thresh, use_mask=False, cam_int=K_t,
        )

    faces = estimator.faces
    if hasattr(faces, 'cpu'):
        faces = faces.detach().cpu().numpy()
    faces = faces.astype(np.int32)

    results = []
    for o in outputs:
        verts  = np.asarray(o['pred_vertices'],    dtype=np.float32)
        joints = np.asarray(o['pred_keypoints_3d'], dtype=np.float32)
        cam_t  = np.asarray(o['pred_cam_t'],       dtype=np.float32)
        results.append({
            'vertices':  verts + cam_t[None, :],
            'faces':     faces,
            'joints_3d': joints,
            'cam_t':     cam_t,
        })
    return results


print('Running SAM3D Body (pedestrians)...')
ped_dets     = sam3_results.get('pedestrian', [])
body_results = run_sam3d_body(
    body_estimator, img_path, img_bgr, K, ped_dets,
    bbox_thresh=BODY_BBOX_THRESH, iou_merge_thresh=SAM3_IOU_MERGE_THRESH,
)
print(f'  {len(body_results)} pedestrian(s) reconstructed.')

# Unload SAM3D Body before loading SAM3D Objects
del body_estimator, _body_model
gc.collect()
torch.cuda.empty_cache()
print(f'SAM3D Body unloaded. GPU free: {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB')

## 6. Pointmap + SAM3D Objects — All Other Class Meshes
SAM3D Objects is loaded once, used for both pointmap computation and per-object inference, then unloaded.

Pointmap mode is controlled by `POINTMAP_MODE` (set in Section 1):

| Mode | Depth source | Coverage | Metric? | Scope |
|---|---|---|---|---|
| `baseline` | MoGe monocular depth | Dense | No (relative) | — |
| `o1_lidar` | Raw LiDAR projection | Sparse (~1–2%) | Yes | Per frame |
| `o2_moge_affine` | MoGe + global LiDAR affine fit | Dense | Yes | Per frame |
| `o3_local_affine` | MoGe + HDBSCAN-filtered per-object affine fit | Dense | Yes | Per object |


In [ ]:
import hdbscan as _hdbscan
from inference import Inference as SAM3DObjectsInference
from pytorch3d.renderer import look_at_view_transform
from pytorch3d.transforms import Transform3d, quaternion_to_matrix

print('Loading SAM3D Objects...')
obj_inference = SAM3DObjectsInference(str(SAM3D_OBJ_CFG), compile=False)
print('  SAM3D Objects ready.')


# ── Baseline: MoGe relative-depth pointmap ────────────────────────────────────

def compute_moge_pointmap(obj_inference, img_rgb, K):
    """
    Run MoGe → dense relative-depth pointmap in PyTorch3D convention (-X_r3, -Y_r3, Z).
    Z is non-metric (relative MoGe units).
    """
    H, W = img_rgb.shape[:2]
    rgba = np.concatenate([img_rgb, np.full((H, W, 1), 255, dtype=np.uint8)], axis=-1)
    with obj_inference._pipeline.device:
        ptmap_dict = obj_inference._pipeline.compute_pointmap(rgba)
    ptmap_p3d = ptmap_dict['pointmap'].cpu().permute(1, 2, 0).numpy().astype(np.float32)
    _fx, _fy, _cx, _cy = K[0,0], K[1,1], K[0,2], K[1,2]
    u_grid, v_grid = np.meshgrid(np.arange(W, dtype=np.float32), np.arange(H, dtype=np.float32))
    Z_map = ptmap_p3d[..., 2]
    X_r3  = (u_grid - _cx) * Z_map / _fx
    Y_r3  = (v_grid - _cy) * Z_map / _fy
    ptmap = np.stack([-X_r3, -Y_r3, Z_map], axis=-1).astype(np.float32)
    return ptmap, Z_map


# ── O1: sparse LiDAR pointmap ─────────────────────────────────────────────────

def compute_lidar_pointmap(pts_cam_vis, u_vis, v_vis, H, W):
    """
    Build a sparse per-pixel LiDAR pointmap in PyTorch3D space.
    Pixels with no LiDAR return are NaN.
    """
    ptmap = np.full((H, W, 3), np.nan, dtype=np.float32)
    u_int = np.round(u_vis).astype(int)
    v_int = np.round(v_vis).astype(int)
    valid = (u_int >= 0) & (u_int < W) & (v_int >= 0) & (v_int < H)
    fx, fy, cx, cy = K[0,0], K[1,1], K[0,2], K[1,2]
    X_cam = (u_vis[valid] - cx) * Z_vis[valid] / fx
    Y_cam = (v_vis[valid] - cy) * Z_vis[valid] / fy
    ptmap[v_int[valid], u_int[valid], 0] = -X_cam
    ptmap[v_int[valid], u_int[valid], 1] = -Y_cam
    ptmap[v_int[valid], u_int[valid], 2] =  Z_vis[valid]
    return ptmap


# ── O2: MoGe + global affine calibration ─────────────────────────────────────

def compute_moge_affine_pointmap(obj_inference, img_rgb, K, u_vis, v_vis, Z_vis,
                                  z_min=0.5, z_max=80.0, min_pts=5):
    """
    O2: dense MoGe depth map calibrated with a single global affine fit
    Z_metric = a*Z_moge + b fitted on all in-image LiDAR returns.
    Returns (ptmap (H,W,3), Z_metric_map (H,W), a_fit, b_fit).
    """
    H, W = img_rgb.shape[:2]
    _, Z_moge_map = compute_moge_pointmap(obj_inference, img_rgb, K)
    u_int = np.round(u_vis).astype(int).clip(0, W-1)
    v_int = np.round(v_vis).astype(int).clip(0, H-1)
    Z_moge_at_lidar = Z_moge_map[v_int, u_int]
    valid = np.isfinite(Z_moge_at_lidar) & (Z_vis > z_min) & (Z_vis < z_max)
    n_fit = int(valid.sum())
    print(f'  Affine fit: {n_fit} LiDAR/MoGe pairs')
    if n_fit < min_pts:
        print('  [WARN] Too few pairs — using unscaled MoGe.')
        a_fit, b_fit = 1.0, 0.0
        Z_metric_map = Z_moge_map.copy()
    else:
        Z_m = Z_moge_at_lidar[valid]; Z_l = Z_vis[valid]
        A = np.stack([Z_m, np.ones_like(Z_m)], axis=1)
        (a_fit, b_fit), _, _, _ = np.linalg.lstsq(A, Z_l, rcond=None)
        Z_pred = a_fit * Z_m + b_fit
        rmse = float(np.sqrt(np.mean((Z_pred - Z_l)**2)))
        r2   = float(1.0 - np.sum((Z_l - Z_pred)**2) / np.sum((Z_l - Z_l.mean())**2))
        print(f'  Z_metric = {a_fit:.4f} x Z_moge + {b_fit:.4f}   R^2={r2:.4f}  RMSE={rmse:.3f} m')
        Z_metric_map = np.maximum(a_fit * Z_moge_map + b_fit, 0.1).astype(np.float32)
    fx, fy, cx, cy = K[0,0], K[1,1], K[0,2], K[1,2]
    u_grid, v_grid = np.meshgrid(np.arange(W, dtype=np.float32), np.arange(H, dtype=np.float32))
    X_r3 = (u_grid - cx) * Z_metric_map / fx
    Y_r3 = (v_grid - cy) * Z_metric_map / fy
    ptmap = np.stack([-X_r3, -Y_r3, Z_metric_map], axis=-1).astype(np.float32)
    return ptmap, Z_metric_map, a_fit, b_fit


# ── O3: per-object local affine calibration with HDBSCAN ─────────────────────

def _fit_affine(Z_moge_vals, Z_lidar_vals, min_pts=4, z_min=0.5, z_max=80.0):
    """Fit Z = a*Z_moge + b via least-squares. Returns (a, b) or None if too few points."""
    valid = np.isfinite(Z_moge_vals) & (Z_lidar_vals > z_min) & (Z_lidar_vals < z_max)
    if valid.sum() < min_pts:
        return None
    Z_m = Z_moge_vals[valid]; Z_l = Z_lidar_vals[valid]
    A = np.stack([Z_m, np.ones_like(Z_m)], axis=1)
    (a, b), _, _, _ = np.linalg.lstsq(A, Z_l, rcond=None)
    return a, b


def _build_ptmap_from_affine(Z_moge_map, K, a, b):
    """Apply Z_metric = a*Z_moge + b to full map and build PyTorch3D pointmap."""
    H, W = Z_moge_map.shape
    fx, fy, cx, cy = K[0,0], K[1,1], K[0,2], K[1,2]
    Z_metric = np.maximum(a * Z_moge_map + b, 0.1).astype(np.float32)
    u_grid, v_grid = np.meshgrid(np.arange(W, dtype=np.float32), np.arange(H, dtype=np.float32))
    X_r3 = (u_grid - cx) * Z_metric / fx
    Y_r3 = (v_grid - cy) * Z_metric / fy
    return np.stack([-X_r3, -Y_r3, Z_metric], axis=-1).astype(np.float32)


def compute_local_affine_ptmap_for_object(Z_moge_map, K,
                                            pts_ego_vis, u_vis, v_vis, Z_vis,
                                            binary_mask, H, W,
                                            min_cluster_size=3, min_samples=1,
                                            cluster_eps=0.5, min_pts=4,
                                            z_min=0.5, z_max=80.0):
    """
    O3: per-object MoGe + HDBSCAN local affine calibration.

    Logic
    -----
    1. Find LiDAR points whose projected pixel falls inside the object's SAM3 mask.
    2. Run HDBSCAN on their 3D ego-frame coordinates to isolate the dominant
       surface cluster — discards ground bleeds, adjacent objects, and noise.
       (Operating in metric 3D space makes the cluster distance threshold
       physically meaningful and range-invariant.)
    3. Fit Z_metric = a*Z_moge + b via least-squares on the clean surface points.
       This (a,b) is specific to this object's depth range.
    4. Apply the fitted transform to the full Z_moge_map and rebuild the XYZ
       pointmap. SAM3D Objects internally crops to the bounding box, so context
       pixels outside the mask also benefit from the calibration.

    Falls back to O2 global affine (all in-image LiDAR) if HDBSCAN finds too
    few clean in-mask points for a reliable fit.

    Parameters
    ----------
    Z_moge_map      : (H, W) float32  pre-computed MoGe relative depth (shared across objects)
    K               : (3, 3) camera intrinsics
    pts_ego_vis     : (M, 3) float32  ego-frame coords of visible LiDAR points
    u_vis, v_vis    : (M,)   float32  pixel coords of visible LiDAR points
    Z_vis           : (M,)   float32  metric depths of visible LiDAR points
    binary_mask     : (H, W) bool     SAM3 mask for this object
    min_cluster_size: HDBSCAN min_cluster_size parameter
    min_pts         : minimum clean points needed for a reliable affine fit

    Returns
    -------
    ptmap : (H, W, 3) float32  metric pointmap in PyTorch3D convention
    a, b  : float  fitted affine coefficients (for logging)
    mode  : str    'local' | 'global_fallback' | 'unscaled_fallback'
    """
    u_int = np.round(u_vis).astype(int).clip(0, W-1)
    v_int = np.round(v_vis).astype(int).clip(0, H-1)

    # Step 1 — find in-mask LiDAR points
    in_mask = binary_mask[v_int, u_int]
    pts_inmask  = pts_ego_vis[in_mask]
    Z_inmask    = Z_vis[in_mask]
    u_inmask    = u_vis[in_mask]
    v_inmask    = v_vis[in_mask]
    Z_moge_all  = Z_moge_map[v_int, u_int]

    if len(pts_inmask) < min_cluster_size:
        # Too few in-mask points even before clustering — go straight to global fallback
        ab = _fit_affine(Z_moge_all, Z_vis, min_pts=min_pts, z_min=z_min, z_max=z_max)
        if ab is None:
            return _build_ptmap_from_affine(Z_moge_map, K, 1.0, 0.0), 1.0, 0.0, 'unscaled_fallback'
        return _build_ptmap_from_affine(Z_moge_map, K, *ab), ab[0], ab[1], 'global_fallback'

    # Step 2 — HDBSCAN in 3D ego space
    clusterer = _hdbscan.HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        metric='euclidean',
        cluster_selection_epsilon=cluster_eps,
    )
    labels = clusterer.fit_predict(pts_inmask)
    unique, counts = np.unique(labels[labels >= 0], return_counts=True)

    if len(unique) == 0:
        # All noise — global fallback
        ab = _fit_affine(Z_moge_all, Z_vis, min_pts=min_pts, z_min=z_min, z_max=z_max)
        if ab is None:
            return _build_ptmap_from_affine(Z_moge_map, K, 1.0, 0.0), 1.0, 0.0, 'unscaled_fallback'
        return _build_ptmap_from_affine(Z_moge_map, K, *ab), ab[0], ab[1], 'global_fallback'

    # Step 3 — keep dominant cluster, fit affine
    dominant = unique[np.argmax(counts)]
    keep = labels == dominant
    u_clean_int = np.round(u_inmask[keep]).astype(int).clip(0, W-1)
    v_clean_int = np.round(v_inmask[keep]).astype(int).clip(0, H-1)
    Z_moge_clean = Z_moge_map[v_clean_int, u_clean_int]
    Z_clean      = Z_inmask[keep]

    ab = _fit_affine(Z_moge_clean, Z_clean, min_pts=min_pts, z_min=z_min, z_max=z_max)
    if ab is None:
        # Not enough clean points — global fallback
        ab = _fit_affine(Z_moge_all, Z_vis, min_pts=min_pts, z_min=z_min, z_max=z_max)
        if ab is None:
            return _build_ptmap_from_affine(Z_moge_map, K, 1.0, 0.0), 1.0, 0.0, 'unscaled_fallback'
        return _build_ptmap_from_affine(Z_moge_map, K, *ab), ab[0], ab[1], 'global_fallback'

    # Step 4 — apply to full frame
    return _build_ptmap_from_affine(Z_moge_map, K, *ab), ab[0], ab[1], 'local'


# ── Dispatch: compute per-frame pointmap (or prep for per-object O3) ──────────

print(f'\nComputing pointmap (mode={POINTMAP_MODE!r})...')

Z_moge_map = None   # populated for O2 and O3; cached across objects in O3

if POINTMAP_MODE == 'o1_lidar':
    ptmap_full = compute_lidar_pointmap(pts_cam_vis, u_vis, v_vis, H, W)
    Z_map      = ptmap_full[..., 2]
    n_lidar_px = int(np.isfinite(Z_map).sum())
    print(f'  LiDAR pointmap: {n_lidar_px:,} annotated pixels '
          f'({100*n_lidar_px/(H*W):.2f}% coverage)')

elif POINTMAP_MODE == 'o2_moge_affine':
    ptmap_full, Z_map, _a, _b = compute_moge_affine_pointmap(
        obj_inference, img_rgb, K, u_vis, v_vis, Z_vis
    )

elif POINTMAP_MODE == 'o3_local_affine':
    # MoGe runs once here; per-object affine happens inside run_sam3d_objects
    print('  Running MoGe (shared base for per-object calibration)...')
    _, Z_moge_map = compute_moge_pointmap(obj_inference, img_rgb, K)
    Z_map      = Z_moge_map          # shown in viz (uncalibrated — calibration is per-object)
    ptmap_full = None                # not used; computed per-object below

else:   # 'baseline'
    print('  Running MoGe (no LiDAR calibration)...')
    ptmap_full, Z_map = compute_moge_pointmap(obj_inference, img_rgb, K)
    print(f'  MoGe Z range: [{np.nanmin(Z_map):.3f}, {np.nanmax(Z_map):.3f}] (relative)')

# ── Visualise ─────────────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(14, 5))
if POINTMAP_MODE == 'o1_lidar':
    ax.imshow(img_rgb)
    sc = ax.scatter(u_vis, v_vis, c=Z_vis, cmap='plasma', s=1, vmin=0, vmax=60, alpha=0.9)
    plt.colorbar(sc, ax=ax, label='Depth Z (m)')
    ax.set_title('O1 — sparse LiDAR pointmap (projected on image)')
elif POINTMAP_MODE == 'o2_moge_affine':
    im = ax.imshow(Z_map, cmap='plasma', vmin=0, vmax=60)
    plt.colorbar(im, ax=ax, label='Metric depth Z (m)')
    ax.scatter(u_vis, v_vis, c='white', s=1, alpha=0.5)
    ax.set_title(f'O2 — MoGe + global affine  (Z = {_a:.3f} x Z_moge + {_b:.3f})\n'
                 f'white dots = LiDAR returns used for fit')
elif POINTMAP_MODE == 'o3_local_affine':
    im = ax.imshow(Z_map, cmap='plasma')
    plt.colorbar(im, ax=ax, label='Relative depth (MoGe units — per-object calibration applied during inference)')
    ax.scatter(u_vis, v_vis, c='white', s=1, alpha=0.5)
    ax.set_title('O3 — MoGe base map (uncalibrated)\n'
                 'per-object HDBSCAN + local affine applied inside SAM3D Objects loop')
else:
    im = ax.imshow(Z_map, cmap='plasma')
    plt.colorbar(im, ax=ax, label='Relative depth (MoGe units)')
    ax.set_title('Baseline — MoGe relative depth map (no LiDAR)')
ax.axis('off')
plt.tight_layout()
plt.show()


# ── SAM3D Objects inference ───────────────────────────────────────────────────

def mesh_to_r3(output):
    """Convert SAM3D Objects output from object-local space to R3 (OpenCV) camera space."""
    from sam3d_objects.data.dataset.tdfy.transforms_3d import compose_transform

    def _sq(t):
        while t.dim() > 2: t = t.squeeze(0)
        return t

    quat  = _sq(output['rotation'].detach().cpu().float())
    trans = _sq(output['translation'].detach().cpu().float())
    scale = _sq(output['scale'].detach().cpu().float())
    if quat.dim() == 1:
        quat = quat.unsqueeze(0)

    l2c = compose_transform(scale=scale, rotation=quaternion_to_matrix(quat), translation=trans)
    R_r3_to_p3d, _ = look_at_view_transform(
        eye=torch.tensor([[0., 0., -1.]]), at=torch.tensor([[0., 0., 0.]]),
        up=torch.tensor([[0., -1., 0.]]),
    )
    t_p3d_to_r3 = Transform3d().rotate(R_r3_to_p3d).inverse()
    mesh_res    = output['mesh'][0]
    verts_local = mesh_res.vertices.float().cpu()
    faces       = mesh_res.faces.long().cpu().numpy().astype(np.int32)
    xyz_p3d     = l2c.transform_points(verts_local.unsqueeze(0)).squeeze(0)
    verts_r3    = t_p3d_to_r3.transform_points(xyz_p3d.unsqueeze(0)).squeeze(0).numpy().astype(np.float32)
    return verts_r3, faces


def run_sam3d_objects(inference_model, img_rgb, sam3_results, prompts_config, ptmap_t=None,
                       o3_data=None):
    """
    Run SAM3D Objects for all non-pedestrian classes.

    Parameters
    ----------
    ptmap_t  : (H, W, 3) float32 tensor — pre-computed full-frame pointmap for
               baseline / O1 / O2.  Passed as-is to every object inference call.
    o3_data  : dict — if provided, O3 mode: compute a per-object calibrated
               pointmap for each detection using local HDBSCAN affine fitting.
               Expected keys: Z_moge_map, K, pts_ego_vis, u_vis, v_vis, Z_vis, H, W.
    """
    results = []
    for prompt, pipeline in prompts_config.items():
        if pipeline != 'objects':
            continue
        dets = sam3_results.get(prompt, [])
        if not dets:
            continue
        print(f'  "{prompt}": {len(dets)} instance(s)...')
        for d in dets:
            try:
                if o3_data is not None:
                    # O3: compute a fresh per-object calibrated pointmap
                    ptmap_obj, a_obj, b_obj, mode = compute_local_affine_ptmap_for_object(
                        Z_moge_map  = o3_data['Z_moge_map'],
                        K           = o3_data['K'],
                        pts_ego_vis = o3_data['pts_ego_vis'],
                        u_vis       = o3_data['u_vis'],
                        v_vis       = o3_data['v_vis'],
                        Z_vis       = o3_data['Z_vis'],
                        binary_mask = d['binary_mask'],
                        H           = o3_data['H'],
                        W           = o3_data['W'],
                    )
                    print(f'    [{mode}]  Z = {a_obj:.4f}*Z_moge + {b_obj:.4f}')
                    ptmap_t_use = torch.tensor(ptmap_obj, dtype=torch.float32)
                else:
                    ptmap_t_use = ptmap_t

                out = inference_model(img_rgb, d['binary_mask'], seed=42, pointmap=ptmap_t_use)
                verts_r3, faces = mesh_to_r3(out)
                results.append({
                    'vertices':    verts_r3,
                    'faces':       faces,
                    'binary_mask': d['binary_mask'],
                    'score':       d['score'],
                    'prompt':      prompt,
                })
            except Exception as e:
                print(f'    [WARN] failed for "{prompt}": {e}')
    return results


# ── Call run_sam3d_objects with the right arguments ──────────────────────────

if POINTMAP_MODE == 'o3_local_affine':
    o3_data = dict(
        Z_moge_map  = Z_moge_map,
        K           = K,
        pts_ego_vis = pts_ego_vis,
        u_vis       = u_vis,
        v_vis       = v_vis,
        Z_vis       = Z_vis,
        H           = H,
        W           = W,
    )
    print('Running SAM3D Objects (O3 — per-object local affine)...')
    obj_results = run_sam3d_objects(
        obj_inference, img_rgb, sam3_results, SAM3_PROMPTS, o3_data=o3_data
    )
else:
    ptmap_t = torch.tensor(ptmap_full, dtype=torch.float32)
    print('Running SAM3D Objects...')
    obj_results = run_sam3d_objects(
        obj_inference, img_rgb, sam3_results, SAM3_PROMPTS, ptmap_t=ptmap_t
    )

print(f'\n{len(obj_results)} object mesh(es) reconstructed.')

# Unload SAM3D Objects
del obj_inference
gc.collect()
torch.cuda.empty_cache()
print(f'SAM3D Objects unloaded. GPU free: {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB')


In [ ]:
# Quick 2D projection of meshes onto image
def project(pts, K):
    """(N,3) camera-space pts → (N,2) pixel coords. Returns -9999 for behind-camera pts."""
    z = pts[:, 2]
    valid = z > 0
    uv = np.full((len(pts), 2), -9999.0, dtype=np.float32)
    uv[valid, 0] = pts[valid, 0] / z[valid] * K[0, 0] + K[0, 2]
    uv[valid, 1] = pts[valid, 1] / z[valid] * K[1, 1] + K[1, 2]
    return uv


vis = img_rgb.copy()
for r in body_results:
    uv = project(r['vertices'], K)
    ok = (uv[:, 0] >= 0) & (uv[:, 0] < W) & (uv[:, 1] >= 0) & (uv[:, 1] < H)
    for u, v in uv[ok].astype(int):
        cv2.circle(vis, (u, v), 1, (255, 80, 80), -1)

obj_colors = [(80, 220, 80), (80, 80, 255), (255, 200, 0), (200, 0, 255), (0, 220, 220)]
for i, r in enumerate(obj_results):
    uv  = project(r['vertices'], K)
    ok  = (uv[:, 0] >= 0) & (uv[:, 0] < W) & (uv[:, 1] >= 0) & (uv[:, 1] < H)
    col = obj_colors[i % len(obj_colors)]
    for u, v in uv[ok].astype(int):
        cv2.circle(vis, (u, v), 1, col, -1)

plt.figure(figsize=(14, 5))
plt.imshow(vis)
plt.axis('off')
plt.title(f'Projected meshes — {len(body_results)} pedestrian(s) [red], {len(obj_results)} object(s) [colour]')
plt.tight_layout()
plt.show()

## 8. Module: Orientation Estimation

- **Pedestrians**: `facing_direction()` — shoulder line → nose-disambiguated unit vector in XZ plane. Fully resolved.
- **Objects**: `None` — 180° yaw ambiguity is accepted for stage 1. PCA long axis only.

The orientation result is a unit vector in camera XZ space for pedestrians, `None` for objects.

In [ ]:
# ── Orientation estimation ────────────────────────────────────────────────────

_NOSE       = 0
_L_SHOULDER = 5
_R_SHOULDER = 6


def facing_direction(joints):
    """
    Estimate horizontal facing direction of a pedestrian from MHR70 skeleton.

    Parameters
    ----------
    joints : (70, 3) float32  MHR70 keypoints in body-relative space

    Returns
    -------
    fwd : (3,) float32  unit vector in XZ plane, Y=0. Fully resolved (no 180° ambiguity).
    """
    shoulder_vec = joints[_R_SHOULDER] - joints[_L_SHOULDER]
    sv_xz = np.array([shoulder_vec[0], 0.0, shoulder_vec[2]], dtype=np.float32)
    sv_xz /= np.linalg.norm(sv_xz) + 1e-8

    # 90° rotation in XZ gives a facing candidate
    fwd = np.array([-sv_xz[2], 0.0, sv_xz[0]], dtype=np.float32)

    # Disambiguate via nose: nose should be in front of the shoulder midpoint
    shoulder_mid = (joints[_L_SHOULDER] + joints[_R_SHOULDER]) / 2.0
    if np.dot(fwd, joints[_NOSE] - shoulder_mid) < 0:
        fwd = -fwd

    return fwd


def estimate_orientation(result, kind):
    """
    Estimate orientation for a single detection.

    Parameters
    ----------
    result : dict from body_results or obj_results
    kind   : 'pedestrian' | 'object'

    Returns
    -------
    fwd : (3,) unit vector in camera XZ plane, or None (ambiguous)
    """
    if kind == 'pedestrian':
        return facing_direction(result['joints_3d'])
    # Objects: ambiguous for stage 1
    return None


# Attach orientation to results
for r in body_results:
    r['orientation_fwd'] = estimate_orientation(r, 'pedestrian')

for r in obj_results:
    r['orientation_fwd'] = estimate_orientation(r, 'object')

print('Orientation estimation done.')
for i, r in enumerate(body_results):
    print(f'  Pedestrian {i}: fwd = {r["orientation_fwd"].round(3)}')
for i, r in enumerate(obj_results):
    print(f'  Object {i} ({r["prompt"]}): orientation = ambiguous (stage 1)')

## 9. Module: OBB Fitting

- **Pedestrians**: `compute_obb_pedestrian()` — projects mesh vertices onto `(right, up, fwd)` axes derived from `facing_direction`. Stays in camera (body-relative) space.
- **Objects**: `compute_obb_gravity_aligned()` — PCA footprint in ego frame. No ground anchoring (stage 1). 180° yaw ambiguity in the long axis.

In [ ]:
# ── OBB Fitting ───────────────────────────────────────────────────────────────

_BBOX_EDGES = [
    (0,1),(1,2),(2,3),(3,0),   # bottom face
    (4,5),(5,6),(6,7),(7,4),   # top face
    (0,4),(1,5),(2,6),(3,7),   # verticals
]


def compute_obb_pedestrian(vertices, fwd):
    """
    Fit an OBB aligned to the pedestrian's facing direction.

    Parameters
    ----------
    vertices : (V, 3) float32  mesh vertices in camera space (R3)
    fwd      : (3,)   float32  unit facing vector in XZ plane (Y=0)

    Returns
    -------
    corners : (8, 3) float32  OBB corners in camera space
    center  : (3,)   float32
    dims    : (3,)   float32  [width, height, depth]
    """
    right = np.array([fwd[2], 0.0, -fwd[0]], dtype=np.float32)   # 90° CW from fwd in XZ
    up    = np.array([0.0, 1.0,  0.0], dtype=np.float32)

    r_proj = vertices @ right
    y_proj = vertices[:, 1]
    f_proj = vertices @ fwd

    r_min, r_max = float(r_proj.min()), float(r_proj.max())
    y_min, y_max = float(y_proj.min()), float(y_proj.max())
    f_min, f_max = float(f_proj.min()), float(f_proj.max())

    corners = np.array([
        r_min*right + y_min*up + f_min*fwd,
        r_max*right + y_min*up + f_min*fwd,
        r_max*right + y_max*up + f_min*fwd,
        r_min*right + y_max*up + f_min*fwd,
        r_min*right + y_min*up + f_max*fwd,
        r_max*right + y_min*up + f_max*fwd,
        r_max*right + y_max*up + f_max*fwd,
        r_min*right + y_max*up + f_max*fwd,
    ], dtype=np.float32)

    center = corners.mean(axis=0)
    dims   = np.array([r_max-r_min, y_max-y_min, f_max-f_min], dtype=np.float32)
    return corners, center, dims


def compute_obb_gravity_aligned(verts_r3, R_c2e, t_c2e, ground_z=None,
                                 height_pct=(1, 99), height_clamp=(0.9, 2.8)):
    """
    Fit a gravity-aligned OBB to mesh vertices, in ego frame.

    Footprint (yaw / length / width) from PCA of mesh XY footprint in ego space.
    Yaw has 180° ambiguity (no LiDAR to resolve it at stage 1).

    Parameters
    ----------
    verts_r3     : (V, 3) float32  mesh vertices in R3 camera space
    R_c2e        : (3, 3) float64  rotation camera → ego
    t_c2e        : (3,)   float64  translation camera → ego
    ground_z     : float | None    ego-frame ground height (None = image-only stage)
    height_pct   : (low, high)     percentiles for robust height estimate
    height_clamp : (min_h, max_h)  plausible height range [m]

    Returns
    -------
    corners_ego : (8, 3) float64  OBB corners in ego frame
    center_ego  : (3,)   float64
    dims        : (3,)   float64  [length, width, height]
    yaw_rad     : float           yaw angle of long axis (±π ambiguous)
    """
    verts_ego = (R_c2e @ verts_r3.T).T + t_c2e

    # PCA on XY footprint
    xy    = verts_ego[:, :2]
    xy_c  = xy - xy.mean(axis=0)
    _, evecs = np.linalg.eigh(np.cov(xy_c.T))
    evecs = evecs[:, ::-1]   # largest eigenvalue first

    ax_l = np.array([evecs[0,0], evecs[1,0], 0.], dtype=np.float64)   # long axis
    ax_w = np.array([evecs[0,1], evecs[1,1], 0.], dtype=np.float64)   # short axis
    ax_h = np.array([0., 0., 1.],                 dtype=np.float64)   # height (ego Z-up)

    # Ensure right-hand system
    if ax_l[0]*ax_w[1] - ax_l[1]*ax_w[0] < 0:
        ax_w = -ax_w
    axes  = np.stack([ax_l, ax_w, ax_h], axis=1)  # (3, 3)

    proj  = verts_ego @ axes
    min_p = proj.min(axis=0).copy()
    max_p = proj.max(axis=0).copy()

    if ground_z is not None:
        vz          = verts_ego[:, 2]
        raw_h       = float(np.percentile(vz, height_pct[1]) - np.percentile(vz, height_pct[0]))
        robust_h    = float(np.clip(raw_h, height_clamp[0], height_clamp[1]))
        min_p[2]    = ground_z
        max_p[2]    = ground_z + robust_h

    mid_p      = (min_p + max_p) / 2.0
    center_ego = axes @ mid_p

    offsets = np.array([
        [min_p[0]-mid_p[0], min_p[1]-mid_p[1], min_p[2]-mid_p[2]],
        [max_p[0]-mid_p[0], min_p[1]-mid_p[1], min_p[2]-mid_p[2]],
        [max_p[0]-mid_p[0], max_p[1]-mid_p[1], min_p[2]-mid_p[2]],
        [min_p[0]-mid_p[0], max_p[1]-mid_p[1], min_p[2]-mid_p[2]],
        [min_p[0]-mid_p[0], min_p[1]-mid_p[1], max_p[2]-mid_p[2]],
        [max_p[0]-mid_p[0], min_p[1]-mid_p[1], max_p[2]-mid_p[2]],
        [max_p[0]-mid_p[0], max_p[1]-mid_p[1], max_p[2]-mid_p[2]],
        [min_p[0]-mid_p[0], max_p[1]-mid_p[1], max_p[2]-mid_p[2]],
    ])
    corners_ego = (axes @ offsets.T).T + center_ego
    dims        = max_p - min_p
    yaw_rad     = float(np.arctan2(evecs[1,0], evecs[0,0]))

    return corners_ego, center_ego, dims, yaw_rad


def fit_obb(result, kind, R_c2e, t_c2e):
    """
    Fit an OBB for a single detection.

    Parameters
    ----------
    result : dict
    kind   : 'pedestrian' | 'object'
    R_c2e  : (3, 3)  camera → ego rotation
    t_c2e  : (3,)    camera → ego translation

    Returns
    -------
    Adds 'obb_corners', 'obb_center', 'obb_dims' (and 'obb_yaw' for objects) to result in-place.
    """
    verts = result['vertices']
    if kind == 'pedestrian':
        fwd = result['orientation_fwd']
        corners, center, dims = compute_obb_pedestrian(verts, fwd)
        result['obb_corners'] = corners    # camera space
        result['obb_center']  = center
        result['obb_dims']    = dims
        result['obb_yaw']     = float(np.arctan2(fwd[0], fwd[2]))  # approx yaw in camera
    else:
        corners, center, dims, yaw = compute_obb_gravity_aligned(
            verts, R_c2e, t_c2e, ground_z=None
        )
        result['obb_corners'] = corners    # ego space
        result['obb_center']  = center
        result['obb_dims']    = dims
        result['obb_yaw']     = yaw        # ±π ambiguous


# Fit OBBs
for r in body_results:
    fit_obb(r, 'pedestrian', R_c2e, t_c2e)

for r in obj_results:
    fit_obb(r, 'object', R_c2e, t_c2e)

print('OBB fitting done.')
for i, r in enumerate(body_results):
    print(f'  Pedestrian {i}: dims (W×H×D) = {r["obb_dims"].round(3)} m (camera-relative)')
for i, r in enumerate(obj_results):
    print(f'  Object {i} ({r["prompt"]}): dims (L×W×H) = {r["obb_dims"].round(3)} m (ego frame, non-metric scale)')

In [ ]:
# ── Cyclist / motorcyclist OBB merge ─────────────────────────────────────────
# When a pedestrian body result (SAM3D Body) and a bicycle or motorcycle result
# (SAM3D Objects) originate from the same physical person-on-vehicle, their
# OBBs are merged:
#   1. The body OBB center (camera space) is transformed to ego space.
#   2. If the distance to a bicycle/motorcycle OBB center (ego space) is < RIDER_MERGE_DIST,
#      vertices from both meshes are combined and a fresh OBB is recomputed.
#   3. The vehicle class label is kept; the pedestrian body entry is removed.
#
# Distance rationale: body OBB center (torso, ~1 m height) vs. bicycle OBB center
# (frame, ~0.5 m height) are typically 0.6-1.0 m apart in 3D ego space even when
# the rider is directly mounted. 1.5 m is tight enough to avoid accidentally
# merging a pedestrian walking next to a parked bike.

RIDER_MERGE_DIST = 1.5   # metres (3D ego space)
RIDER_CLASSES    = {'bicycle', 'motorcycle'}


def merge_rider_obbs(body_res, obj_res, R_c2e, t_c2e, dist_thresh=RIDER_MERGE_DIST):
    """
    Merge pedestrian body OBBs with co-located bicycle / motorcycle OBBs.
    Returns (new_body_res, obj_res) — matched body entries are removed.
    """
    body_used = set()
    for bi, br in enumerate(body_res):
        # body obb_center is in camera space -> transform to ego
        bc_ego = R_c2e @ br['obb_center'].astype(np.float64) + t_c2e

        best_oi, best_d = None, float('inf')
        for oi, or_ in enumerate(obj_res):
            if or_['prompt'] not in RIDER_CLASSES:
                continue
            d = float(np.linalg.norm(bc_ego - np.array(or_['obb_center'], dtype=np.float64)))
            if d <= dist_thresh and d < best_d:
                best_d, best_oi = d, oi

        if best_oi is None:
            continue

        or_      = obj_res[best_oi]
        combined = np.concatenate([br['vertices'], or_['vertices']], axis=0).astype(np.float32)
        corners, center, dims, yaw = compute_obb_gravity_aligned(
            combined, R_c2e, t_c2e, ground_z=None
        )
        or_['vertices']    = combined
        or_['obb_corners'] = corners
        or_['obb_center']  = center
        or_['obb_dims']    = dims
        or_['obb_yaw']     = yaw
        body_used.add(bi)
        print(f'  [rider merge] pedestrian -> "{or_["prompt"]}"  (d={best_d:.2f} m)')

    return [br for bi, br in enumerate(body_res) if bi not in body_used], obj_res


n_body_before = len(body_results)
body_results, obj_results = merge_rider_obbs(body_results, obj_results, R_c2e, t_c2e)
n_merged = n_body_before - len(body_results)
print(f'Rider merge: {n_merged} pedestrian(s) merged into vehicle OBB(s).')
print(f'  Body results: {n_body_before} -> {len(body_results)}')
print(f'  Obj  results: {len(obj_results)} (OBBs expanded for merged entries)')


## 10. 2D OBB Visualization
Project OBB wireframes onto the image.

In [ ]:
def draw_obb_on_image(img_rgb, corners, K, color, R_c2e=None, t_c2e=None, in_ego=False):
    """
    Draw an OBB wireframe on an image by projecting 8 corners.

    Parameters
    ----------
    corners   : (8, 3)  OBB corners — camera space OR ego space
    in_ego    : bool    if True, transform corners ego → camera first
    """
    vis = img_rgb.copy()
    c3d = corners.astype(np.float64)
    if in_ego and R_c2e is not None:
        # ego → camera: R_c2e^T @ (p - t_c2e)
        c3d = (R_c2e.T @ (c3d - t_c2e).T).T
    uv = project(c3d.astype(np.float32), K)
    for e0, e1 in _BBOX_EDGES:
        p0, p1 = uv[e0], uv[e1]
        if (p0 == -9999).any() or (p1 == -9999).any():
            continue
        pt0 = (int(np.clip(p0[0], 0, W-1)), int(np.clip(p0[1], 0, H-1)))
        pt1 = (int(np.clip(p1[0], 0, W-1)), int(np.clip(p1[1], 0, H-1)))
        cv2.line(vis, pt0, pt1, color, 2)
    return vis


vis = img_rgb.copy()
for r in body_results:
    vis = draw_obb_on_image(vis, r['obb_corners'], K, color=(255, 60, 60), in_ego=False)

for r in obj_results:
    vis = draw_obb_on_image(vis, r['obb_corners'], K, color=(60, 220, 60),
                             R_c2e=R_c2e, t_c2e=t_c2e, in_ego=True)

plt.figure(figsize=(14, 5))
plt.imshow(vis)
plt.axis('off')
plt.title('OBBs — pedestrians [red, camera space] | objects [green, ego space]')
plt.tight_layout()
plt.show()

## 11. 3D Visualization (plotly)
Interactive scene with meshes and OBBs in ego frame. All pedestrian geometry is first converted from camera to ego space.

In [ ]:
def cam_to_ego(pts, R_c2e, t_c2e):
    """Transform (N,3) points from camera (R3) to ego frame."""
    return (R_c2e @ pts.T).T + t_c2e


OBJ_COLORS = [
    'rgb(64,221,64)', 'rgb(64,64,255)', 'rgb(255,200,0)',
    'rgb(200,0,255)', 'rgb(0,221,221)', 'rgb(255,100,0)',
]

traces = []


def _obb_lines(corners, color):
    """Scatter3d trace for one OBB wireframe (12 edges, None-separated)."""
    xs, ys, zs = [], [], []
    for e0, e1 in _BBOX_EDGES:
        xs += [corners[e0, 0], corners[e1, 0], None]
        ys += [corners[e0, 1], corners[e1, 1], None]
        zs += [corners[e0, 2], corners[e1, 2], None]
    return go.Scatter3d(x=xs, y=ys, z=zs, mode='lines',
                        line=dict(color=color, width=3), showlegend=False)


# ── Pedestrian meshes (red) ───────────────────────────────────────────────────
for i, r in enumerate(body_results):
    verts_ego = cam_to_ego(r['vertices'].astype(np.float64), R_c2e, t_c2e)
    f = r['faces']
    traces.append(go.Mesh3d(
        x=verts_ego[:, 0], y=verts_ego[:, 1], z=verts_ego[:, 2],
        i=f[:, 0], j=f[:, 1], k=f[:, 2],
        color='rgb(255,64,64)', opacity=0.7,
        name=f'pedestrian_{i}', showlegend=True,
    ))

# ── Pedestrian OBBs (dark red wireframe) ─────────────────────────────────────
for i, r in enumerate(body_results):
    corners_ego = cam_to_ego(r['obb_corners'].astype(np.float64), R_c2e, t_c2e)
    traces.append(_obb_lines(corners_ego, 'rgb(170,0,0)'))

# ── Pedestrian facing arrows (yellow) ────────────────────────────────────────
for i, r in enumerate(body_results):
    center_ego = cam_to_ego(r['obb_center'].reshape(1, 3).astype(np.float64), R_c2e, t_c2e)[0]
    fwd_ego    = R_c2e @ r['orientation_fwd'].astype(np.float64)
    tip_ego    = center_ego + fwd_ego * 0.5
    traces.append(go.Scatter3d(
        x=[center_ego[0], tip_ego[0]],
        y=[center_ego[1], tip_ego[1]],
        z=[center_ego[2], tip_ego[2]],
        mode='lines', line=dict(color='rgb(255,255,0)', width=6),
        name=f'ped_fwd_{i}', showlegend=False,
    ))

# ── Object meshes + OBBs (coloured) ──────────────────────────────────────────
for i, r in enumerate(obj_results):
    col       = OBJ_COLORS[i % len(OBJ_COLORS)]
    verts_ego = cam_to_ego(r['vertices'].astype(np.float64), R_c2e, t_c2e)
    f         = r['faces']
    traces.append(go.Mesh3d(
        x=verts_ego[:, 0], y=verts_ego[:, 1], z=verts_ego[:, 2],
        i=f[:, 0], j=f[:, 1], k=f[:, 2],
        color=col, opacity=0.7,
        name=f'{r["prompt"]}_{i}', showlegend=True,
    ))
    traces.append(_obb_lines(r['obb_corners'].astype(np.float64), col))

fig = go.Figure(data=traces)
fig.update_layout(
    title='Auto-labeling Stage 1 — ego frame',
    scene=dict(
        xaxis_title='X ego', yaxis_title='Y ego', zaxis_title='Z ego',
        aspectmode='data',
    ),
    margin=dict(l=0, r=0, b=0, t=40),
)
fig.show()

## 12. BEV Visualization — LiDAR + Pseudo Labels + Ground Truth
Interactive bird's-eye-view plot. LiDAR coloured by height, pseudo-label OBBs as solid outlines, GT annotations as white dashed boxes.

> **Note:** GT annotations exist only on keyframes. If the selected frame is not a keyframe the nearest preceding keyframe's GT is used.

In [ ]:
# cam_sd, pts_ego, lidar_token are already loaded in Section 3.5.

# ── GT annotations ────────────────────────────────────────────────────────────
# GT exists only on keyframes — walk back through sample_data to the nearest one.
gt_boxes_ego = []
_token = cam_sd_token
while _token:
    _sd = nusc.get('sample_data', _token)
    if _sd['is_key_frame']:
        _sample_gt      = nusc.get('sample', _sd['sample_token'])
        _lidar_gt_token = _sample_gt['data']['LIDAR_TOP']
        _ego_gt         = nusc.get('ego_pose',
                              nusc.get('sample_data', _lidar_gt_token)['ego_pose_token'])
        R_e2g_gt = Quaternion(_ego_gt['rotation']).rotation_matrix.astype(np.float64)
        t_e2g_gt = np.array(_ego_gt['translation'], dtype=np.float64)

        for ann_token in _sample_gt['anns']:
            ann        = nusc.get('sample_annotation', ann_token)
            center_g   = np.array(ann['translation'], dtype=np.float64)
            half_w     = ann['size'][0] / 2.0   # size = [width, length, height]
            half_l     = ann['size'][1] / 2.0
            # Global → ego
            center_e   = R_e2g_gt.T @ (center_g - t_e2g_gt)
            # Yaw in ego frame
            R_box_e    = R_e2g_gt.T @ Quaternion(ann['rotation']).rotation_matrix
            yaw        = float(np.arctan2(R_box_e[1, 0], R_box_e[0, 0]))
            cy, sy     = np.cos(yaw), np.sin(yaw)
            corners_2d = center_e[:2] + np.array([
                [+half_l*cy - half_w*sy, +half_l*sy + half_w*cy],
                [+half_l*cy + half_w*sy, +half_l*sy - half_w*cy],
                [-half_l*cy + half_w*sy, -half_l*sy - half_w*cy],
                [-half_l*cy - half_w*sy, -half_l*sy + half_w*cy],
            ])
            gt_boxes_ego.append({
                'corners_2d': corners_2d,
                'category':   ann['category_name'],
            })
        print(f'GT: {len(gt_boxes_ego)} annotation(s) from keyframe {_sd["sample_token"][:8]}...')
        break
    _token = _sd['prev'] if _sd['prev'] else ''

if not gt_boxes_ego and not _token:
    print('GT: no preceding keyframe found.')

# ── BEV plot ──────────────────────────────────────────────────────────────────
_rng  = np.random.default_rng(0)
_idx  = _rng.choice(len(pts_ego), min(40_000, len(pts_ego)), replace=False)
pts_d = pts_ego[_idx]
_mask = (np.abs(pts_d[:, 0]) < 60) & (np.abs(pts_d[:, 1]) < 30)
pts_d = pts_d[_mask]

bev_traces = []

# LiDAR points coloured by height
bev_traces.append(go.Scatter(
    x=pts_d[:, 0], y=pts_d[:, 1],
    mode='markers',
    marker=dict(size=1.5, color=pts_d[:, 2], colorscale='Viridis',
                cmin=-2, cmax=3, showscale=True,
                colorbar=dict(title='Height (m)', thickness=12, x=1.01)),
    name='LiDAR', hoverinfo='skip',
))

# GT boxes — white dashed, grouped by top-level category
_GT_COLORS = {
    'vehicle':        'rgb(255,255,255)',
    'human':          'rgb(255,210,100)',
    'movable_object': 'rgb(180,180,180)',
    'static_object':  'rgb(140,140,140)',
}
_shown_gt = set()
for b in gt_boxes_ego:
    top_cat = b['category'].split('.')[0]
    color   = _GT_COLORS.get(top_cat, 'rgb(200,200,200)')
    name    = f'GT {top_cat}'
    c       = b['corners_2d']
    bev_traces.append(go.Scatter(
        x=[c[0,0], c[1,0], c[2,0], c[3,0], c[0,0]],
        y=[c[0,1], c[1,1], c[2,1], c[3,1], c[0,1]],
        mode='lines',
        line=dict(color=color, width=2, dash='dash'),
        name=name, legendgroup=name,
        showlegend=(name not in _shown_gt),
    ))
    _shown_gt.add(name)

# Pseudo-label OBBs — solid, coloured by class
_PSEUDO_COLORS = {
    'pedestrian':           'rgb(255,80,80)',
    'car':                  'rgb(80,220,80)',
    'bicycle and cyclist':  'rgb(80,120,255)',
    'motorcycle':           'rgb(255,160,0)',
    'bus':                  'rgb(200,80,255)',
    'truck':                'rgb(0,210,210)',
    'construction vehicle': 'rgb(255,80,180)',
    'trailer':              'rgb(160,220,60)',
}

def _footprint_xy(corners_xy):
    """Sort corners by angle from centroid to get a correct closed polygon."""
    cx, cy  = corners_xy.mean(axis=0)
    angles  = np.arctan2(corners_xy[:, 1] - cy, corners_xy[:, 0] - cx)
    return corners_xy[np.argsort(angles)]

_shown_pseudo = set()
for r in body_results:
    corners_e = cam_to_ego(r['obb_corners'].astype(np.float64), R_c2e, t_c2e)
    foot      = _footprint_xy(corners_e[:, :2])
    color     = _PSEUDO_COLORS['pedestrian']
    name      = 'pseudo pedestrian'
    bev_traces.append(go.Scatter(
        x=list(foot[:, 0]) + [foot[0, 0]],
        y=list(foot[:, 1]) + [foot[0, 1]],
        mode='lines', line=dict(color=color, width=2),
        name=name, legendgroup=name,
        showlegend=(name not in _shown_pseudo),
    ))
    _shown_pseudo.add(name)

for r in obj_results:
    foot  = _footprint_xy(r['obb_corners'].astype(np.float64)[:, :2])
    color = _PSEUDO_COLORS.get(r['prompt'], 'rgb(200,200,200)')
    name  = f'pseudo {r["prompt"]}'
    bev_traces.append(go.Scatter(
        x=list(foot[:, 0]) + [foot[0, 0]],
        y=list(foot[:, 1]) + [foot[0, 1]],
        mode='lines', line=dict(color=color, width=2),
        name=name, legendgroup=name,
        showlegend=(name not in _shown_pseudo),
    ))
    _shown_pseudo.add(name)

# Ego vehicle marker
bev_traces.append(go.Scatter(
    x=[0], y=[0], mode='markers',
    marker=dict(symbol='arrow-up', angle=90, size=16,
                color='yellow', line=dict(color='black', width=1)),
    name='ego vehicle', showlegend=True,
))

fig_bev = go.Figure(data=bev_traces)
fig_bev.update_layout(
    title=(f'BEV — Scene {SCENE_IDX}, frame {FRAME_IDX} | '
           f'solid = pseudo labels | dashed = GT'),
    xaxis=dict(title='X ego (forward, m)', range=[-10, 60],
               scaleanchor='y', scaleratio=1,
               showgrid=True, gridcolor='rgba(255,255,255,0.08)', zeroline=False),
    yaxis=dict(title='Y ego (left, m)', range=[-30, 30],
               showgrid=True, gridcolor='rgba(255,255,255,0.08)', zeroline=False),
    plot_bgcolor='rgb(15,15,25)',
    paper_bgcolor='rgb(15,15,25)',
    font=dict(color='white'),
    legend=dict(bgcolor='rgba(0,0,0,0.5)', font=dict(size=11)),
    width=1000, height=580,
    margin=dict(l=0, r=80, t=40, b=0),
)
fig_bev.show()

## 12. Summary
Print a quick summary table of all detections.

In [ ]:
print(f'Scene {SCENE_IDX}, frame {FRAME_IDX} — {CAMERA}')
print(f'{"Class":<25} {"Dims (m)":>30} {"Yaw (rad)":>12} {"Score":>8}')
print('-' * 80)

for i, r in enumerate(body_results):
    d = r['obb_dims']
    print(f'{"pedestrian_" + str(i):<25} W={d[0]:.2f} H={d[1]:.2f} D={d[2]:.2f} (camera) '
          f'{r["obb_yaw"]:>10.3f}  {"—":>8}')

for i, r in enumerate(obj_results):
    d     = r['obb_dims']
    label = r['prompt'] + '_' + str(i)
    print(f'{label:<25} L={d[0]:.2f} W={d[1]:.2f} H={d[2]:.2f} (ego)   '
          f'{r["obb_yaw"]:>10.3f}* {r["score"]:>8.3f}')

print('\n* yaw has 180° ambiguity for objects at stage 1 (no LiDAR)')